# Retail Sales Analysis (Sample Superstore Subset)

This notebook analyzes a curated subset of retail sales data. It demonstrates a clean analytics workflow: data cleaning, feature engineering, exploratory analysis, business questions, and visualization.


## 1. Setup
We import core libraries and configure a consistent plotting style.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)


## 2. Configuration
We define the input dataset path in a single variable for easy reuse.


In [ ]:
DATA_PATH = "data/retail_sales_sample.csv"


## 3. Helper Functions
We refactor the workflow into small functions to keep the notebook readable and reusable.


In [ ]:
def load_sales_data(file_path: str) -> pd.DataFrame:
    '''Load raw sales data from a CSV file.'''
    return pd.read_csv(file_path)


def clean_sales_data(raw_df: pd.DataFrame) -> pd.DataFrame:
    '''Clean raw sales data and return a standardized DataFrame.'''
    cleaned_df = raw_df.copy()

    # Trim whitespace in text fields
    text_columns = [
        "customer_name",
        "segment",
        "region",
        "state",
        "category",
        "sub_category",
        "product",
    ]
    for column in text_columns:
        cleaned_df[column] = cleaned_df[column].astype(str).str.strip()

    # Normalize casing for categorical fields
    cleaned_df["segment"] = cleaned_df["segment"].str.title()
    cleaned_df["region"] = cleaned_df["region"].str.title()
    cleaned_df["category"] = cleaned_df["category"].str.title()
    cleaned_df["sub_category"] = cleaned_df["sub_category"].str.title()

    # Parse dates safely
    cleaned_df["order_date"] = pd.to_datetime(cleaned_df["order_date"], errors="coerce")
    cleaned_df["ship_date"] = pd.to_datetime(cleaned_df["ship_date"], errors="coerce")

    # Clean numeric fields
    cleaned_df["sales"] = (
        cleaned_df["sales"]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
    )
    cleaned_df["sales"] = pd.to_numeric(cleaned_df["sales"], errors="coerce")

    def parse_discount(value) -> float:
        if pd.isna(value):
            return np.nan
        value_str = str(value).strip()
        if value_str.endswith("%"):
            return float(value_str.replace("%", "")) / 100
        return float(value_str)

    cleaned_df["discount"] = cleaned_df["discount"].apply(parse_discount).fillna(0)
    cleaned_df["quantity"] = pd.to_numeric(cleaned_df["quantity"], errors="coerce")
    cleaned_df["profit"] = pd.to_numeric(cleaned_df["profit"], errors="coerce")

    # Remove exact duplicates
    cleaned_df = cleaned_df.drop_duplicates()

    return cleaned_df


def add_features(cleaned_df: pd.DataFrame) -> pd.DataFrame:
    '''Add derived features for analysis.'''
    enriched_df = cleaned_df.copy()

    enriched_df["order_month"] = enriched_df["order_date"].dt.to_period("M").astype(str)
    enriched_df["ship_days"] = (enriched_df["ship_date"] - enriched_df["order_date"]).dt.days
    enriched_df["profit_margin"] = enriched_df["profit"] / enriched_df["sales"]

    enriched_df["sales_bucket"] = pd.cut(
        enriched_df["sales"],
        bins=[0, 50, 200, 1000],
        labels=["Small", "Medium", "Large"],
        include_lowest=True,
    )

    return enriched_df


def summarize_by_category(enriched_df: pd.DataFrame) -> pd.DataFrame:
    '''Summarize sales and profit by category.'''
    return (
        enriched_df.groupby("category")[["sales", "profit"]]
        .sum()
        .sort_values("sales", ascending=False)
    )


def summarize_by_region(enriched_df: pd.DataFrame) -> pd.Series:
    '''Summarize sales by region.'''
    return enriched_df.groupby("region")["sales"].sum().sort_values(ascending=False)


def summarize_by_segment(enriched_df: pd.DataFrame) -> pd.DataFrame:
    '''Summarize sales and profit by segment.'''
    return (
        enriched_df.groupby("segment")[["sales", "profit"]]
        .sum()
        .sort_values("sales", ascending=False)
    )


def top_customers_by_sales(enriched_df: pd.DataFrame, top_n: int = 10) -> pd.Series:
    '''Return the top N customers by total sales.'''
    return (
        enriched_df.groupby(["customer_id", "customer_name"])["sales"]
        .sum()
        .sort_values(ascending=False)
        .head(top_n)
    )


def monthly_sales_trend(enriched_df: pd.DataFrame) -> pd.Series:
    '''Return total sales by order month.'''
    return enriched_df.groupby("order_month")["sales"].sum()


## 4. Load Data
We load the raw CSV and preview the first rows.


In [ ]:
raw_sales_df = load_sales_data(DATA_PATH)
raw_sales_df.head()


## 5. Data Cleaning
We standardize text, parse dates, clean numeric fields, and remove duplicates.


In [ ]:
clean_sales_df = clean_sales_data(raw_sales_df)
clean_sales_df.isna().sum()


## 6. Feature Engineering
We add derived columns to support business analysis.


In [ ]:
analysis_df = add_features(clean_sales_df)
analysis_df.head()


## 7. Exploratory Analysis
A quick statistical overview to understand distributions and ranges.


In [ ]:
analysis_df[["sales", "quantity", "discount", "profit", "ship_days"]].describe()


## 8. Business Questions
We compute the main aggregations used to answer leadership questions.


In [ ]:
category_summary = summarize_by_category(analysis_df)
category_summary


In [ ]:
region_summary = summarize_by_region(analysis_df)
region_summary


In [ ]:
segment_summary = summarize_by_segment(analysis_df)
segment_summary


In [ ]:
monthly_sales = monthly_sales_trend(analysis_df)
monthly_sales


In [ ]:
top_customers = top_customers_by_sales(analysis_df, top_n=10)
top_customers


In [ ]:
discount_profit_correlation = analysis_df[["discount", "profit_margin"]].corr().loc["discount", "profit_margin"]
discount_profit_correlation


## 9. Visualizations
Charts that highlight category performance, trends, and discount effects.


In [ ]:
category_summary.plot(kind="bar", y="sales", legend=False)
plt.title("Total Sales by Category")
plt.ylabel("Sales")
plt.xlabel("Category")
plt.show()


In [ ]:
monthly_sales.plot(marker="o")
plt.title("Monthly Sales Trend")
plt.ylabel("Sales")
plt.xlabel("Order Month")
plt.xticks(rotation=45)
plt.show()


In [ ]:
sns.scatterplot(data=analysis_df, x="discount", y="profit_margin", hue="category")
plt.title("Discount vs Profit Margin")
plt.show()


In [ ]:
sns.boxplot(data=analysis_df, x="segment", y="sales")
plt.title("Sales Distribution by Segment")
plt.show()


## 10. Key Insights

**Category Performance**
- Technology leads sales and profits, driven by phones and accessories.

**Discount Impact**
- Higher discounts correlate with lower profit margins, indicating margin pressure.

**Customer Value**
- Consumer and Corporate segments are the top revenue drivers.

**Trend**
- Sales trend upward into March, showing early-year momentum.

**Concentration**
- A small set of customers contributes a large share of total revenue.


## 11. How To Run

1. Install dependencies:
   - `python3 -m pip install pandas numpy matplotlib seaborn jupyter`
2. Execute the notebook:
   - `python3 -m nbconvert --execute --to notebook --inplace sales_analysis.ipynb`
3. Open the notebook in Jupyter or VS Code to review outputs and charts.
